# Known landmark localization in $\mathbb{R}^3$

This example demonstrates 3D localization of a single point state `wX` from four known landmarks using `KnownLandmarkSquaredRangeFactor3`. 

A single unknown point $x \in \mathbb{R}^3$ is observed through range measurements to several fixed, known landmarks $L_i$. Each factor contributes the scalar residual:

$$ e_i(x) = \|x - L_i\|^2 - (d_i)^2, $$

where $d_i$ is the measured range to landmark $L_i$. Several geometrically distinct landmarks are required to uniquely determine the point.

See the `KnownLandmarkSquaredRangeFactor` [API notebook](../../../gtsam/slam/doc/KnownLandmarkSquaredRangeFactor.ipynb) for more details on the factor. 

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation, Atlanta, Georgia 30332-0415
All Rights Reserved
Authors: Frank Dellaert, et al. (see THANKS for the full author list)
See LICENSE for the license information

In [ ]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import gtsam

## Create a 3D synthetic localization problem
We choose one true 3D point and four fixed landmarks. The landmarks are spread through space so that the resulting range measurements provide independent geometric constraints.

Gaussian noise is added to each raw range measurement for realism.

In [ ]:
rng = np.random.default_rng(42)

x_true = np.array([1.0, 2.0, 3.0])

landmarks = [
    np.array([0.0, 0.0, 0.0]),
    np.array([5.0, 0.0, 0.0]),
    np.array([0.0, 6.0, 0.0]),
    np.array([1.0, 0.0, 8.0]),
]

range_sigma = 0.10  # raw range standard deviation [m]

true_ranges = [np.linalg.norm(x_true - wL) for wL in landmarks]

measurement_noise = rng.normal(loc=0.0, scale=range_sigma, size=len(landmarks))

measured_ranges = true_ranges + measurement_noise

print("True point:", x_true)
print(f"Raw range standard deviation: {range_sigma:.3f} m\n")

for i, (wL, r_true, r_measured) in enumerate(zip(landmarks, true_ranges, measured_ranges)):
    print(
        f"L{i}: {wL} "
        f"True range = {r_true:.3f} m, ",
        f"Measured range = {r_measured:.3f} m"
    )

## Propagate range uncertainty into squared-range units
The sensor uncertainty above is expressed in range units, while `KnownLandmarkSquaredRangeFactor3` uses the squared-range residual
$$
e_i(x) = \|x - L_i\|^2 - d_i^2.
$$
For $y = d^2$, first-order uncertainty propagation gives
$$
\sigma_y \approx \left|\frac{\partial d^2}{\partial d}\right|\sigma_d = 2d\,\sigma_d.
$$
Therefore, the standard deviation used by each factor is approximately
$$
\sigma_{d^2} \approx 2d\,\sigma_d.
$$
Even when each range measurement has the same $\sigma_d$, their respective squared-range uncertainties differ between landmarks.

## Build the factor graph
Each range measurement contributes one factor to the same unknown point state.

In [ ]:
key = 0
graph = gtsam.NonlinearFactorGraph()

for wL, measured_range in zip(landmarks, measured_ranges):
    sigma_squared_range = 2.0 * measured_range * range_sigma
    model = gtsam.noiseModel.Isotropic.Sigma(1, sigma_squared_range)
    graph.add(
        gtsam.KnownLandmarkSquaredRangeFactor3(key, wL, measured_range, model)
        )

## Choose an initial estimate
The nonlinear optimizer needs an initial value for the point state. The initial position is deliberately offset from the true position.

In [ ]:
x_initial = np.array([1.2, 1.7, 3.2])

initial = gtsam.Values()
initial.insert(key, x_initial)

## Optimize
We solve the nonlinear least-squares problem with Levenberg-Marquardt.

In [ ]:
optimizer = gtsam.LevenbergMarquardtOptimizer(graph, initial)
result = optimizer.optimize()

x_estimated = result.atVector(key)

print("True point: ", x_true)
print("Initial point: ", x_initial)
print("Estimated point: ", x_estimated)
print("Final graph error: ", graph.error(result))
print("Localization error: ", np.linalg.norm(x_estimated - x_true))

## Visualize the result

In [ ]:
landmarks_array = np.vstack(landmarks)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection = "3d")

ax.scatter(
    landmarks_array[:, 0],
    landmarks_array[:, 1],
    landmarks_array[:, 2],
    marker="x",
    s=80,
    label="Known Landmarks",
)

ax.scatter(*x_true, marker="*", s=180, label="True Position")

ax.scatter(*x_initial, marker="^", s=100, label="Initial Estimate")

ax.scatter(*x_estimated, marker="o", s=100, label="Estimated Position")

ax.plot(
    [x_true[0], x_estimated[0]],
    [x_true[1], x_estimated[1]],
    [x_true[2], x_estimated[2]],
    linewidth=2,
)

for i, wL in enumerate(landmarks):
    ax.text(*wL, f"L{i}")

ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_zlabel("z [m]")
ax.set_box_aspect((1, 1, 1))
ax.set_title("Known-Landmark Squared-Range Localization")
ax.legend()

plt.show()

## Takeaways
- `KnownLandmarkSquaredRangeFactor3` estimates a 3D Euclidean point, not a pose.
- Each factor relates the point to one fixed known landmark.
- Pass the non-negative range measurement $d$ to the constructor; the factor stores $d^2$ internally. 
- The residual is scalar, so each factor uses a one-dimensional noise model.
- Raw range uncertainty should be propagated into squared-range units before use in the factor noise model.
- Known landmarks are fixed factor data and are not inserted into `Values`.